# S1.2 — 진단 run 정밀판: 역전 + proxy 예측 (W4/3/2)

**S1 대비 개편 (엔진 코드감사 기반):**
- **loss-R + acc-R 동시 측정** — proxy는 loss-곡률량이라 loss 공간서 검증해야 apples-to-apples (S1의 acc-R↔dtHd 불일치 해소).
- **고정 1200스텝 수렴앵커** — S1의 적응형 plateau(eps 0.1)가 느린 마라톤층을 조기 절단 → dtHd 약화의 주범. 전 층 동일 예산 공정비교.
- **촘촘한 t-grid {10,30,100,300,700,1200}** — ρ-vs-t 교차(λ²↔λ)를 실제로 해상.
- **순열검정 p값(주) + 부트스트랩 CI(보조)** — 자의적 SNR>3 컷 대신 방어가능한 유의성.
- **HVP n_batches=8** (normHd2 평균 편향↓), **λ_max 파워이터** (η·λ_max<1 = 닫힌형태 타당성).

**규모:** W4/3/2 × 전 층(21) × seed{0,1,2} × t 6점. 출력 `outputs/s1_2/`·`figures/s1_2/` 전용(기존 무손상). ~8–10h.

In [ ]:
# --- Colab 셋업 ---
import os
REPO = '/content/26_Capstone'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- 엔진 + Drive + 경로 (S1.2 전용 출력) ---
from qat_engine import *
import numpy as np, matplotlib.pyplot as plt

try:
    from google.colab import drive; drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'
for sub in ['checkpoints','outputs','figures','outputs/s1_2','figures/s1_2']:
    os.makedirs(f'{ART}/{sub}', exist_ok=True)
DATA_ROOT = f'{ART}/data'
CKPT      = f'{ART}/checkpoints/resnet18_cifar100_fp32.pt'
OUTDIR    = f'{ART}/outputs/s1_2'
FIG       = f'{ART}/figures/s1_2'
OUT       = f'{OUTDIR}/s1_2_runs.jsonl'
print('device', DEVICE, '| ART', ART, '| S1.2 out', OUTDIR)

In [ ]:
# --- S1.2 설정 ---
BITS          = [4, 3, 2]                       # 주력 / 빈무대 / 붕괴앵커 (W8 = 기존 인용)
SEEDS         = [0, 1, 2]                        # 부트스트랩 보조; 주 판정 = 순열검정(21층)
GRID          = (10, 30, 100, 300, 700, 1200)   # 고정 t 체크포인트
STEPS         = 1200                            # 고정 수렴앵커 (적응형 plateau 폐기)
SHORT, CONV   = 30, 1200                         # 단기 vs 수렴 비교점
LR            = 1e-3
N_BATCH_PROXY = 8                               # HVP/Fisher 추정 배치 (normHd2 편향↓)
print('bits', BITS, '| seeds', SEEDS, '| grid', GRID, '| short/conv', SHORT, CONV)

## baseline + 층 목록 + λ_max (lr 타당성)

In [ ]:
train_loader, val_loader, calib_loader = get_loaders('cifar100', batch=128, calib_size=512, data_root=DATA_ROOT)
assert os.path.exists(CKPT), 'FP32 체크포인트 없음 — S0 먼저'
fp_model = load_model('resnet18', 'cifar100', ckpt=CKPT)
fp_acc = evaluate(fp_model, val_loader)
layers = list_quant_layers(make_ptq_model(fp_model, 4))
costs  = get_layer_costs(make_ptq_model(fp_model, 4), layers)
print(f'FP32 천장 {fp_acc:.2f} | 양자화 층 {len(layers)}개')

# λ_max 점검 (대표 3층 @ W4): eta*|lambda_max| < 1 이어야 닫힌형태 (1-eta*lambda)^t 단조
pm4 = make_ptq_model(fp_model, 4)
rep_layers = [layers[1], layers[len(layers)//2], layers[-1]]
print(f'--- lr 타당성: eta*|lambda_max| (eta={LR:.0e}) ---')
for L in rep_layers:
    lam = lambda_max_layer(pm4, L, calib_loader, n_iter=12, n_batches=4)
    flag = 'OK(<1)' if LR*abs(lam) < 1 else 'WARN(>=1)'
    print(f'  {L:24s} lambda_max~{lam:+.3e}  eta*|lam|={LR*abs(lam):.3e}  {flag}')

## 스모크 — 3층·1seed·W4 (loss-R / 고정 grid 경로 점검, ~몇 분)
여기서 에러나면 엔진 고치고 git pull 후 재실행. 통과하면 full sweep로.

In [ ]:
sm_layers = [layers[1], layers[len(layers)//2], layers[-1]]
pm = make_ptq_model(fp_model, 4)
ba, bl = evaluate_full(pm, val_loader)
print(f'W4 PTQ acc {ba:.2f} (gap {fp_acc-ba:.2f}%p) | loss {bl:.4f}')
px = proxy_scores(pm, fp_model, 4, calib_loader, layers=sm_layers, n_batches=N_BATCH_PROXY)
for L in sm_layers:
    m = make_ptq_model(fp_model, 4)              # PTQ 베이스라인 ba/bl 재사용(비트당 동일)
    set_trainable(m, [L])
    r = short_qat(m, train_loader, val_loader, steps=STEPS, eval_at=GRID, plateau=False,
                  seed=0, lr=LR, momentum=0.0, track_loss=True)
    racc  = {t: round(r[t]['acc'] - ba, 3) for t in r}
    rloss = {t: round(bl - r[t]['loss'], 4) for t in r}
    print(f'{L}\n   acc-R ={racc}\n   loss-R={rloss}  normHd2={px[L]["normHd2"]:.2e} dtHd={px[L]["dtHd"]:.2e}')
print('스모크 OK — full sweep로')

## 전체 sweep (W4/3/2 × 전층 × seed × 고정 grid) — Drive `outputs/s1_2/` 증분 저장. ~8–10h.

In [ ]:
RESULTS = {}
import json as _json
for bit in BITS:
    pm = make_ptq_model(fp_model, bit)
    pa, pl = evaluate_full(pm, val_loader); gap = fp_acc - pa
    proxies = proxy_scores(pm, fp_model, bit, calib_loader, layers=layers, n_batches=N_BATCH_PROXY)
    _json.dump(proxies, open(f'{OUTDIR}/s1_2_proxies_w{bit}.json', 'w'))
    Racc = {L: {} for L in layers}; Rloss = {L: {} for L in layers}
    for L in layers:
        for s in SEEDS:
            m = make_ptq_model(fp_model, bit)        # PTQ 베이스라인 pa/pl 재사용(비트당 동일·결정적)
            set_trainable(m, [L])
            r = short_qat(m, train_loader, val_loader, steps=STEPS, eval_at=GRID, plateau=False,
                          seed=s, lr=LR, momentum=0.0, track_loss=True)
            for t in r:
                Racc[L].setdefault(t, []).append(r[t]['acc'] - pa)
                Rloss[L].setdefault(t, []).append(pl - r[t]['loss'])
            rec = {**{f'acc_{t}': r[t]['acc'] - pa for t in r}, **{f'loss_{t}': pl - r[t]['loss'] for t in r}}
            log_run({'phase': 'S1.2', 'bit': bit, 'layer': L, 'seed': s}, rec, path=OUT)
    RESULTS[bit] = dict(gap=gap, ptq_acc=pa, ptq_loss=pl, proxies=proxies, Racc=Racc, Rloss=Rloss)
    print(f'[W{bit}] gap {gap:.2f}%p, ptq_loss {pl:.3f} — {len(layers)}층 x {len(SEEDS)}seed 완료')
print('sweep 완료')

## 분석 + 그림 (loss-R 주, acc-R 보조)

In [ ]:
# 헬퍼: 회복행렬(metric 선택) · 신호검증 · breakdown · 유의성(순열p+부트CI) · 수렴평탄성
def Rmat(bit, metric):
    return RESULTS[bit]['Racc'] if metric == 'acc' else RESULTS[bit]['Rloss']
def mean_R(bit, t, metric='loss'):
    M = Rmat(bit, metric); return [float(np.mean(M[L][t])) for L in layers]
def std_R(bit, t, metric='loss'):
    M = Rmat(bit, metric); return [float(np.std(M[L][t])) for L in layers]
def per_seed(bit, t, metric='loss'):
    M = Rmat(bit, metric); return [[M[L][t][s] for L in layers] for s in range(len(SEEDS))]
def proxy_vec(bit, key):
    return [RESULTS[bit]['proxies'][L][key] for L in layers]

def signal_check(bit, t, metric='loss'):
    means = mean_R(bit, t, metric); noise = float(np.mean(std_R(bit, t, metric)))
    ps = per_seed(bit, t, metric)
    pairs = [spearman(ps[i], ps[j]) for i in range(len(SEEDS)) for j in range(i+1, len(SEEDS))]
    rs = float(np.nanmean(pairs)) if pairs else float('nan')
    return dict(spread=float(np.std(means)), seed_noise=noise, snr=float(np.std(means))/max(noise,1e-12), rank_stability=rs)

def is_breakdown(bit):
    return RESULTS[bit]['ptq_acc'] < 5.0

def inversion_report(bit, metric='loss'):
    sh = mean_R(bit, SHORT, metric); cv = mean_R(bit, CONV, metric)
    perm = perm_pvalue_related(sh, cv)
    boot = bootstrap_inversion(per_seed(bit, SHORT, metric), per_seed(bit, CONV, metric))
    c30 = signal_check(bit, SHORT, metric); cpl = signal_check(bit, CONV, metric)
    noise_inv = 1.0 - min(c30['rank_stability'], cpl['rank_stability'])
    real = bool((not is_breakdown(bit)) and (perm['p_value'] < 0.05) and (boot['ci_lo'] > noise_inv))
    return dict(inv=boot['inversion'], ci=(boot['ci_lo'], boot['ci_hi']), perm_rho=perm['rho'], perm_p=perm['p_value'],
                noise_inv=noise_inv, breakdown=is_breakdown(bit), real_inversion=real)
print('헬퍼: Rmat/mean_R/std_R/per_seed/proxy_vec/signal_check/is_breakdown/inversion_report')

In [ ]:
# FIG0 — 신호검증 (loss-R): 층별 회복 mean±std(seed) + SNR/rank-stab (기술통계, 참고)
chk = [b for b in BITS if RESULTS[b]['gap'] > 0.5]
fig, axs = plt.subplots(len(chk), 2, figsize=(12, 4*len(chk)), squeeze=False)
for r_, bit in enumerate(chk):
    for c_, t in enumerate([SHORT, CONV]):
        m = np.array(mean_R(bit, t, 'loss')); sd = np.array(std_R(bit, t, 'loss')); o = np.argsort(m)[::-1]
        ck = signal_check(bit, t, 'loss')
        ax = axs[r_][c_]; ax.errorbar(range(len(layers)), m[o], yerr=sd[o], fmt='o', ms=3, capsize=2)
        ax.set_xlabel('layer (sorted)'); ax.set_ylabel('loss-recovery')
        ax.set_title(f'W{bit} t={t}  SNR={ck["snr"]:.1f} rank-stab={ck["rank_stability"]:.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_signal_check.png', dpi=120); plt.show()
print('SNR/rank-stab = 기술통계. 역전 판정은 순열 p + 부트 CI (아래).')

In [ ]:
# 수렴 평탄성 — |loss-R(1200) - loss-R(700)| 작아야 '수렴앵커=1200' 타당
fig, ax = plt.subplots(figsize=(7,4))
for bit in [b for b in BITS if RESULTS[b]['gap'] > 0.5]:
    d = np.abs(np.array(mean_R(bit, 1200, 'loss')) - np.array(mean_R(bit, 700, 'loss')))
    ax.plot(sorted(d, reverse=True), marker='o', ms=3, label=f'W{bit} (max Δ={d.max():.3f})')
ax.set_xlabel('layer (sorted)'); ax.set_ylabel('|loss-R(1200) - loss-R(700)|'); ax.legend()
ax.set_title('convergence flatness (작을수록 1200=수렴 타당)')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_flatness.png', dpi=120); plt.show()
print('큰 층이 있으면 그 층은 1200서도 아직 수렴 전 — 해석 시 주의.')

In [ ]:
# 역전 (RQ2) — loss-R, 순열 p + 부트 CI. rank 1=best
from scipy.stats import rankdata
def best_rank(v): return len(v) + 1 - rankdata(v)
inv_bits = [b for b in BITS if RESULTS[b]['gap'] > 0.5]
fig, axs = plt.subplots(1, len(inv_bits), figsize=(5*len(inv_bits), 4.8), squeeze=False)
for c_, bit in enumerate(inv_bits):
    rep = inversion_report(bit, 'loss')
    sh = mean_R(bit, SHORT, 'loss'); cv = mean_R(bit, CONV, 'loss')
    ax = axs[0][c_]; ax.scatter(best_rank(sh), best_rank(cv), s=25)
    lim = [0, len(layers)+1]; ax.plot(lim, lim, '--', color='gray', lw=1)
    ax.set_xlabel(f'short (t={SHORT}) rank'); ax.set_ylabel(f'converged (t={CONV}) rank')
    tag = 'BREAKDOWN' if rep['breakdown'] else ('REAL inv' if rep['real_inversion'] else 'n.s.')
    ax.set_title(f'W{bit} inv={rep["inv"]:.2f} CI[{rep["ci"][0]:.2f},{rep["ci"][1]:.2f}]\nperm p={rep["perm_p"]:.3f}  {tag}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_inversion.png', dpi=120); plt.show()

In [ ]:
# proxy↔회복 산점도 (loss-R, 주력 W4)
bit = 4
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
sh = mean_R(bit, SHORT, 'loss'); nh = proxy_vec(bit, 'normHd2')
axs[0].scatter(nh, sh, s=25); axs[0].set_xscale('log')
axs[0].set_xlabel('||H d||^2 (short)'); axs[0].set_ylabel(f'loss-R({SHORT})'); axs[0].set_title(f'short proxy  rho={spearman(nh,sh):.2f}')
cv = mean_R(bit, CONV, 'loss'); dh = proxy_vec(bit, 'dtHd')
axs[1].scatter(dh, cv, s=25)
axs[1].set_xlabel('dT H d (converged)'); axs[1].set_ylabel(f'loss-R({CONV})'); axs[1].set_title(f'converged proxy  rho={spearman(dh,cv):.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_proxy_scatter_w{bit}.png', dpi=120); plt.show()

In [ ]:
# ρ-vs-t 교차 (RQ4 핵심): normHd2(단기) vs dtHd(수렴)이 t 따라 역전하나 — loss-R
fig, axs = plt.subplots(1, len(inv_bits), figsize=(5*len(inv_bits), 4.2), squeeze=False)
for c_, bit in enumerate(inv_bits):
    nh = proxy_vec(bit, 'normHd2'); dh = proxy_vec(bit, 'dtHd')
    rs = [spearman(nh, mean_R(bit, t, 'loss')) for t in GRID]
    rc = [spearman(dh, mean_R(bit, t, 'loss')) for t in GRID]
    ax = axs[0][c_]; ax.plot(GRID, rs, marker='o', label='||Hd||^2 (short)'); ax.plot(GRID, rc, marker='s', label='dT H d (conv)')
    ax.set_xscale('log'); ax.set_xlabel('t'); ax.set_ylabel('rho(proxy, loss-R)'); ax.set_title(f'W{bit}: which proxy wins when'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_rho_vs_t.png', dpi=120); plt.show()
print('교차(normHd2 단기우세 -> dtHd 수렴우세)면 lambda^2<->lambda 직접 증거.')

In [ ]:
# RQ4 — 5 proxy x t x {loss,acc} 순위상관, 저장
PROX = ['normHd2', 'dtHd', 'fisher', 'werr', 'iso_out']
import csv as _csv
for bit in inv_bits:
    print(f'=== W{bit}  rho(proxy, loss-R) ===')
    rows = []
    for k in PROX:
        pv = proxy_vec(bit, k)
        rl = [spearman(pv, mean_R(bit, t, 'loss')) for t in GRID]
        ra = [spearman(pv, mean_R(bit, t, 'acc')) for t in GRID]
        rows.append([k] + [round(x, 3) if not np.isnan(x) else None for x in rl + ra])
        print(f'  {k:9s}', [round(x, 2) if not np.isnan(x) else None for x in rl])
    with open(f'{OUTDIR}/s1_2_proxy_rho_w{bit}.csv', 'w', newline='') as f:
        w = _csv.writer(f); w.writerow(['proxy'] + [f'loss_{t}' for t in GRID] + [f'acc_{t}' for t in GRID]); w.writerows(rows)
print('저장 s1_2_proxy_rho_w*.csv (loss & acc 둘 다)')

In [ ]:
# RQ6 — iso_out vs 회복 (loss-R, 수렴). 음수=출력변화 작은 층이 더 회복
bit = 4; io = proxy_vec(bit, 'iso_out'); rp = mean_R(bit, CONV, 'loss'); rho = spearman(io, rp)
plt.figure(figsize=(6.5, 4.5)); plt.scatter(io, rp, s=25); plt.xscale('log')
plt.xlabel('isolated ||d logit||^2'); plt.ylabel(f'loss-R({CONV})'); plt.title(f'W{bit} RQ6 rho={rho:.2f}')
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_rq6_w{bit}.png', dpi=120); plt.show()
print(f'RQ6 rho={rho:.3f} ->', 'NEG(재현)' if rho < 0 else 'POS/0(반박·무관)')

In [ ]:
# 갭·역전 vs 비트 (W4/3/2 + 기존 W8은 인용)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(BITS, [RESULTS[b]['gap'] for b in BITS], marker='o'); axs[0].set_xlabel('bit'); axs[0].set_ylabel('PTQ gap'); axs[0].set_title('gap vs bit'); axs[0].invert_xaxis()
for bit in inv_bits:
    rep = inversion_report(bit, 'loss'); col = 'red' if rep['breakdown'] else ('C0' if rep['real_inversion'] else 'orange')
    yerr = [[max(rep['inv']-rep['ci'][0], 0)], [max(rep['ci'][1]-rep['inv'], 0)]]
    axs[1].errorbar([bit], [rep['inv']], yerr=yerr, fmt='o', color=col, capsize=4)
axs[1].set_xlabel('bit'); axs[1].set_ylabel('inversion (loss-R) +/-95%CI'); axs[1].set_title('inversion vs bit (blue=real, orange=n.s., red=breakdown)'); axs[1].invert_xaxis()
plt.tight_layout(); plt.savefig(f'{FIG}/s1_2_vs_bit.png', dpi=120); plt.show()

In [ ]:
# 요약 저장 — s1_2_summary.json (loss-R 기준, 순열 p + 부트 CI 포함) + recovery csv
import json as _json, csv as _csv
def _safe(v): return None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), 4)
summary = {'fp_acc': fp_acc,
           'config': dict(bits=BITS, seeds=SEEDS, grid=list(GRID), steps=STEPS, short=SHORT, conv=CONV, n_batch_proxy=N_BATCH_PROXY),
           'bits': {}}
for bit in BITS:
    rep = inversion_report(bit, 'loss') if RESULTS[bit]['gap'] > 0.5 else None
    c30 = signal_check(bit, SHORT, 'loss'); cpl = signal_check(bit, CONV, 'loss')
    summary['bits'][str(bit)] = dict(
        gap=round(RESULTS[bit]['gap'], 3), ptq_acc=round(RESULTS[bit]['ptq_acc'], 2), ptq_loss=round(RESULTS[bit]['ptq_loss'], 4),
        snr_short=_safe(c30['snr']), snr_conv=_safe(cpl['snr']),
        rank_stab_short=_safe(c30['rank_stability']), rank_stab_conv=_safe(cpl['rank_stability']),
        breakdown=bool(is_breakdown(bit)),
        inversion=_safe(rep['inv']) if rep else None,
        inv_ci_lo=_safe(rep['ci'][0]) if rep else None, inv_ci_hi=_safe(rep['ci'][1]) if rep else None,
        perm_rho=_safe(rep['perm_rho']) if rep else None, perm_p=_safe(rep['perm_p']) if rep else None,
        noise_inv=_safe(rep['noise_inv']) if rep else None,
        real_inversion=bool(rep['real_inversion']) if rep else None,
        rho_short_normHd2=_safe(spearman(proxy_vec(bit, 'normHd2'), mean_R(bit, SHORT, 'loss'))),
        rho_conv_dtHd=_safe(spearman(proxy_vec(bit, 'dtHd'), mean_R(bit, CONV, 'loss'))))
    for metric in ['loss', 'acc']:
        with open(f'{OUTDIR}/s1_2_recovery_{metric}_w{bit}.csv', 'w', newline='') as f:
            w = _csv.writer(f); w.writerow(['layer'] + [f't{t}' for t in GRID] + [f'std{t}' for t in GRID])
            for L in layers:
                w.writerow([L] + [round(np.mean(Rmat(bit, metric)[L][t]), 4) for t in GRID] + [round(np.std(Rmat(bit, metric)[L][t]), 4) for t in GRID])
_json.dump(summary, open(f'{OUTDIR}/s1_2_summary.json', 'w'), indent=2, ensure_ascii=False)
print('저장 s1_2_summary.json + recovery csv')
for b in summary['bits']:
    s = summary['bits'][b]
    print(f"W{b} | gap {s['gap']} | inv {s['inversion']} CI[{s['inv_ci_lo']},{s['inv_ci_hi']}] | perm_p {s['perm_p']} | REAL {s['real_inversion']} | breakdown {s['breakdown']}")

In [ ]:
# 비교 — 기존 S1(W8/W4/W2, acc·적응형) vs S1.2(W4/3/2, loss·고정·순열p). 읽기전용
import json as _json
prev = f'{ART}/outputs/s1_summary.json'; new = f'{OUTDIR}/s1_2_summary.json'
print('[기존 S1 — acc-R, 적응형 plateau, SNR컷]')
if os.path.exists(prev):
    for b, s in sorted(_json.load(open(prev))['bits'].items(), key=lambda x: -int(x[0])):
        print(f"  W{b} gap {s['gap']} inv {s.get('inversion_strength')} TRUST {s.get('inversion_trustworthy')} breakdown {s.get('breakdown')}")
else:
    print('  (s1_summary.json 없음)')
print('[S1.2 — loss-R, 고정1200, 순열p + 부트CI]')
for b, s in sorted(_json.load(open(new))['bits'].items(), key=lambda x: -int(x[0])):
    print(f"  W{b} gap {s['gap']} inv {s['inversion']} CI[{s['inv_ci_lo']},{s['inv_ci_hi']}] perm_p {s['perm_p']} REAL {s['real_inversion']} breakdown {s['breakdown']}")

## S1.2 완료 — 무엇이 달라졌나 (S1 대비)

1. **dtHd가 살아났나?** `s1_2_rho_vs_t` 에서 dtHd가 수렴쪽(t=1200)서 normHd2를 추월하면 → S1의 "dtHd 약함"이 *plateau 절단 탓*이었음 확정 + λ²↔λ 교차 직접 증거(척추).
2. **역전이 REAL인가?** `s1_2_summary` 의 `real_inversion` = (붕괴 아님) ∧ (순열 p<0.05) ∧ (부트 CI 하한 > noise_inv=1−rank_stab). True면 자의적 컷 없이 역전 확보.
3. **loss vs acc:** `s1_2_proxy_rho_w*.csv` 에서 loss-R rho가 acc-R rho보다 깨끗하면 → 공간 일치 효과 확인.
4. **수렴 타당성:** flatness 그림서 700→1200 평탄하면 1200=수렴 OK. 안 평탄한 층은 주의.
5. **W2:** breakdown=True면 여전히 붕괴앵커(역전 아님).

**판정 로직(자의적 컷 제거):** 강한 역전(ρ→0)은 검정력 부족으로 보수적 n.s. 처리될 수 있음(과대주장 방지). → 결과(`s1_2_summary.json`·`s1_2_proxy_rho_w*.csv`·그림들) 갖고 와서 해석.